## Context Window: Why Chatbots Forget

Every model has a **context window limit** - the maximum number of tokens it can process in a single request.

This includes input + output tokens combined.

Let's see what happens when you exceed it!

In [5]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='./.env')
API_KEY = os.environ["GEMINI_API_KEY"]

# from google.colab import userdata
# API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)

In [2]:
'python '*10

'python python python python python python python python python python '

In [7]:
# Create a VERY long message to hit the limit
long_story = "Once upon a time, there was a programmer who loved Python. "*10  # Suceed


#long_story = "Once upon a time, there was a programmer who loved Python. "*100000  # ERROR

print(f"📏 Story length: {len(long_story):,} characters")
print(f"📏 Estimated tokens: ~{len(long_story) // 4:,}\n")

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=long_story,
        config={"max_output_tokens": 100}
    )
    print("✅ Request succeeded!")
    print(f"📊 Tokens used: {response.usage_metadata.total_token_count:,}")
    print(f"\n💡 Gemini 3.1 Flash has a HUGE context window (~1M tokens)")
    print(f"   So this request fits comfortably!")
except Exception as e:
    print(f"❌ ERROR: {e}")
    print("\n💡 This is what happens when you exceed the context window!")

📏 Story length: 590 characters
📏 Estimated tokens: ~147

✅ Request succeeded!
📊 Tokens used: 228

💡 Gemini 3.1 Flash has a HUGE context window (~1M tokens)
   So this request fits comfortably!


### What is Context Window?

**Context Window = Maximum tokens in a single request**

Different models have different limits:
- **Gemini 2.5 Flash**: ~1 million tokens
- **GPT-4**: ~128K tokens
- **Claude 4**: ~200K tokens




![image-2.png](attachment:image-2.png)


### Context Window in Conversations

In chat applications, the context window includes:
- **All previous messages** (entire conversation history)
- **Current message**
- **Response**

As conversations grow, tokens accumulate!

In [ ]:
messages = []

def chat(user_message):
    """Send message and track tokens"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=messages
    )

    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )

    # Show token usage
    total_tokens = response.usage_metadata.total_token_count
    print(f"🤖: {response.text}")
    print(f"📊 Total tokens used: {total_tokens} (includes ALL {len(messages)} messages)\n")

    return response.text

# Start conversation
print("👤: Hi! My name is Yash")
chat("Hi! My name is Yash")

print("👤: I'm 25 years old")
chat("I'm 25 years old")

print("👤: I love Python programming")
chat("I love Python programming")

print("👤: I work as a software engineer")
chat("I work as a software engineer")

print("👤: What's my name?")
chat("What's my name?")

👤: Hi! My name is Yash
🤖: Hi Yash! It's nice to meet you. How can I help you today?
📊 Total tokens used: 25 (includes ALL 2 messages)

👤: I'm 25 years old
🤖: Great to know, Yash! Is there anything specific you'd like to talk about or anything I can assist you with? I'm here to help with information, creative tasks, or just to chat.
📊 Total tokens used: 77 (includes ALL 4 messages)

👤: I love Python programming
🤖: That's fantastic, Yash! Python is an incredibly versatile and popular programming language. What do you love about it specifically? Are you working on any personal projects, or do you use it for work or studies? I'd be interested to hear what excites you about Python!
📊 Total tokens used: 139 (includes ALL 6 messages)

👤: I work as a software engineer
🤖: That's excellent, Yash! Working as a software engineer must be very rewarding. How long have you been in the field, and what kind of projects do you typically work on with Python? Are you involved in web development, data scie

'Your name is Yash! 😊'

### The Problem: Tokens Keep Growing

Notice how tokens increase with each message?

In a real long conversation:
- Message 1: 50 tokens
- Message 10: 500 tokens  
- Message 50: 2,500 tokens
- Message 100: 5,000 tokens
- Message 1000: 50,000 tokens

**Eventually, you'll hit the context window limit!**

### Solution: Remove Old Messages

When approaching the limit, remove oldest messages.

**This is why chatbots "forget"!**

In [ ]:
MAX_MESSAGES = 6  # Keep only last 6 messages (3 exchanges)

def chat_with_limit(user_message):
    """Chat with context window management"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    # Remove old messages if too many
    if len(messages) > MAX_MESSAGES:
        removed = messages.pop(0)  # Remove oldest
        print(f"🗑️  Removed old message: {removed.parts[0].text[:50]}...\n")

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=messages
    )

    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )

    print(f"🤖: {response.text}")
    print(f"📊 Messages in memory: {len(messages)}\n")

    return response.text

# Reset and try again
messages = []

print("👤: My favorite color is blue")
chat_with_limit("My favorite color is blue")

print("👤: I have a dog named Max")
chat_with_limit("I have a dog named Max")

print("👤: I live in Mumbai")
chat_with_limit("I live in Mumbai")

print("👤: I enjoy hiking")
chat_with_limit("I enjoy hiking")

# This will forget the first message!
print("👤: What's my favorite color?")
chat_with_limit("What's my favorite color?")

👤: My favorite color is blue
🤖: That's a wonderful favorite color! Blue is such a versatile and popular choice. It can evoke so many different feelings and associations.

What is it about blue that you love the most? For example, do you prefer:

*   **Bright, vibrant blues** like electric blue or turquoise?
*   **Deep, calming blues** like navy or royal blue?
*   **Soft, muted blues** like sky blue or powder blue?
*   **Cool, icy blues**?

Tell me more about your blue! I'd love to hear what makes it your favorite. 😊
📊 Messages in memory: 2

👤: I have a dog named Max
🤖: That's fantastic! Max is a great name for a dog, full of energy and friendliness.

Tell me about Max! I'd love to know:

*   **What kind of dog is Max?** (Breed, mix?)
*   **How old is Max?**
*   **What's his personality like?** (Is he playful, cuddly, mischievous, energetic, calm?)
*   **What are some of his favorite things to do?** (Playing fetch, going for walks, napping, chewing toys?)
*   **Any funny or heartwarming

"That's a fun question! Since I'm an AI, I don't have access to your personal preferences, so I can't know your favorite color.\n\nHowever, I can take a guess based on some general associations, or you can tell me!\n\n**Would you like me to guess, or would you prefer to tell me your favorite color?**\n\nIf I were to guess, I might think of colors associated with things people often enjoy:\n\n*   **Blue:** Often associated with calmness, the sky, and the ocean (which you might appreciate living near Mumbai!).\n*   **Green:** Connected to nature, growth, and tranquility (especially relevant if you enjoy hiking!).\n*   **Yellow:** Cheerful, energetic, and sunny.\n*   **Red:** Passionate, bold, and vibrant.\n\nSo, what is your favorite color? I'm eager to know!"

In [ ]:
chat_with_limit("What is a special about Mango?")

🗑️  Removed old message: That's fantastic! Max is a great name for a dog, f...

🤖: Ah, mangoes! They are truly special, and for so many reasons. Here's why mangoes are so celebrated and what makes them stand out:

**1. Incredible Taste and Aroma:**
*   **Sweetness and Tanginess:** Mangoes offer a unique balance of intense sweetness with a delightful tangy undertone, depending on the variety.
*   **Exotic Flavor Profile:** Their flavor is complex and rich, often described as a blend of peach, pineapple, and citrus.
*   **Unforgettable Aroma:** The fragrance of a ripe mango is captivating and instantly evokes thoughts of summer and tropical bliss.

**2. Variety and Diversity:**
*   **Hundreds of Varieties:** There are literally hundreds, if not thousands, of mango varieties worldwide, each with its own distinct size, shape, color, texture, and flavor. This means there's a mango for almost every palate!
*   **Regional Specialties:** Famous varieties like the Alphonso (often called the "Ki

'Ah, mangoes! They are truly special, and for so many reasons. Here\'s why mangoes are so celebrated and what makes them stand out:\n\n**1. Incredible Taste and Aroma:**\n*   **Sweetness and Tanginess:** Mangoes offer a unique balance of intense sweetness with a delightful tangy undertone, depending on the variety.\n*   **Exotic Flavor Profile:** Their flavor is complex and rich, often described as a blend of peach, pineapple, and citrus.\n*   **Unforgettable Aroma:** The fragrance of a ripe mango is captivating and instantly evokes thoughts of summer and tropical bliss.\n\n**2. Variety and Diversity:**\n*   **Hundreds of Varieties:** There are literally hundreds, if not thousands, of mango varieties worldwide, each with its own distinct size, shape, color, texture, and flavor. This means there\'s a mango for almost every palate!\n*   **Regional Specialties:** Famous varieties like the Alphonso (often called the "King of Mangoes" in India), Kesar, Dasheri, Langra, and Haden have loyal 

### Key Takeaways

1. **Context Window** = Maximum tokens in a single request (input + output)

2. **In conversations**, tokens grow with each message

3. **Must remove old messages** when approaching limit

4. **This causes "forgetting"** - AI loses old context

5. **Exceeding limit = Error** - Request will fail

### Real-World Solutions

- **Summarize** old messages instead of removing
- **Store** important info separately (database)
- **Prioritize** recent messages
- **Use RAG** (Retrieval Augmented Generation) for long-term memory
- **Monitor token usage** and manage proactively